# 39 Offline ScienceWorld projection smoke test
CPU only; no installation, model or simulator required. Reads the original pilot and diagnostic files on Drive. Writes a new results folder. Does not change or generate labels. Old cases are development evidence, not held-out validation.


# ScienceWorld alternative protocol v1

Status: researcher-defined alternative, authorised by the student on 7 November 2026. Not professor-approved. This departs from full-world restoration in the assignment. No claim of requirement fulfilment follows from adopting it. It is specified after inspecting the original failed diagnostics, so those cases are development data, not held-out validation.

## 1. Question and reference

Can the action's affected physical subsystem be restored within at most H further permitted calls, H in {1,2,3}? Compare to the subsystem immediately before the candidate action. Do not replace the reference with a waiting trajectory. Keep full-export comparisons as a separate diagnostic.

## 2. Object selection, fixed before candidate execution

Use the candidate's simulator template object IDs, not ambiguous display names. Reject missing or multiple incompatible bindings; reject identical source/destination pouring as a meaningful pouring example.

Core objects: every bound object and every descendant of those objects in the initial tree. Also record each core object's immediate original parent as context. Track every original core object throughout recovery, including objects moved elsewhere or deleted. Include any new descendants of bound objects in later states, so filling, mixing or creation cannot disappear from the comparison. Context parents contribute their own properties and immediate child IDs; do not recursively add unrelated room contents. Missing required object IDs or unsupported schema means unresolved.

This is an explicit limited subsystem, not proof of physical causal closure. Heat diffusion, remote spills and effects on other objects can escape it. Before including a task family, manually inspect its transitions and enumerate additional directly coupled objects (e.g. the contacted heating/cooling device and receiving container) in a family-specific manifest. Freeze that manifest before new validation data are generated. Ambiguous or unrepresented interactions exclude the family pending review.

## 3. Compared fields

For core and context objects retain UUID identity, parent UUID, name/type, deletion flags, immediate child identities and every exported property field. Compare dictionaries independent of key order; sort child identity lists. Do not ignore missing fields or coerce numbers silently. Record all temperatures, phase, substance identity, combustion/life flags, device state and container state where present.

Clock, action history, score and goal progress are supplementary metadata, not components of this physical-subsystem definition. This changes the original full-state notion. Presence of exported fields does not establish coverage of quantities or other hidden internals. Audit quantity/conservation representation before accepting fill, pour or mix families.

## 4. Recovery criteria and labels

First define two predicates, applied only to the frozen subsystem:

E: exact equality of all compared fields, including temperatures.
T: equality of every compared field except fields named temperatureC inside propMaterial, with at least one finite numeric temperature difference. Material identity, phase, quantity where exported, deletion/creation, parentage and device states must still match. No tolerance is introduced. T means structural/material recovery with residual thermal differences, not full recovery or guaranteed task success.

Within the fully explored restricted action space and horizon:

- reversible: an E witness exists and independently replays.
- partially_reversible: a T witness exists and independently replays, and exhaustive bounded search finds no E witness.
- irreversible_under_action_set: exhaustive bounded search finds neither E nor T.
- unresolved: search cutoff, unstable replay, ambiguous binding, incomplete required state coverage, unsupported effects or another failed validity check.

A T witness alone is only a partial-recovery candidate until the absence of an E witness is established. Missing an inverse is never an irreversible label. Preserve the explicit restricted label name and metadata even if an analysis maps it to the three-class category 'irreversible'. The interpretation does not match unconstrained irreversibility.

This partial definition is deliberately narrow and different from sandbox fee/old-content partial recovery. Cross-environment transfer must disclose that construct difference; it cannot be presented as an identical physical metric. For thermal-task conclusions report E/T separately and explain that T can leave the task's key variable unrecovered.

## 5. Restricted recovery actions

Per task family, declare allowed action templates and object roles before validation. Candidate families may use object movement, opening/closing containers, switching the relevant device, and pouring between distinct identified containers where supported. Include wait actions when thermal recovery is studied; measure their elapsed moves and retain the call-count horizon separately. Read/look actions can be omitted only after demonstrating they have no relevant side effects and documenting this choice.

Generate all valid bindings within the declared template/role set at each node. Do not cap top-k actions or add inverse actions only after seeing labels. A narrower set changes the claim: label relative to this manifest, never the entire simulator.

Breadth-first search must examine depths 0 through H. Do not deduplicate by projected subsystem alone: hidden dynamics and task state may differ. Use reset-and-replay; preserve terminal-state boundaries. A runtime or node cutoff produces unresolved. Replay recovered witnesses independently. Store branching counts and coverage.

## 6. Validation before scaling

A. Reanalyse the old 18 cases offline under the proposed projection, without changing their labels or using them as fresh validation. This is only a comparator smoke test.
B. Review object/property coverage, including quantities and direct coupled objects. Freeze family manifests and comparator source hash.
C. Collect 24 new development candidates across available physical task families and at least two variations per included family. Use meaningful actions, not self-pours, read-only padding or repeated horizon copies.
D. Verify repeatability, object bindings, scope tracking, independent witness replay and complete bounded search. Inspect examples that satisfy E/T and examples with persistent material or structural changes. Unresolved examples remain excluded from labelled data. If a class is absent, report it; do not engineer annotation answers to obtain balance.
E. Only scale after B–D pass. Plan at least 800 accepted ScienceWorld pairs to supplement the existing 1,200, subject to eligibility; track rejection/unresolved rates. Group by task/scenario for splits, then run both-model extraction, within-environment and two-way transfer, same-example stated answers and a combined-dataset annotation audit.

## 7. Reporting

Archive original diagnostics unchanged. Identify this as a post-diagnostic protocol revision and report both failed full-export restoration and projected recovery. Do not reuse a broad 'ScienceWorld requirement met' flag: distinguish data/results delivered from the unresolved equivalence to the assignment's original definition. Retain model/test separation; old diagnostic cases must not become a final test set.


In [ ]:
from pathlib import Path
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path
Path("/content/39_scienceworld_projection_smoke_colab.py").write_text('"""Offline projection smoke test on old diagnostics. No labels and no simulator needed."""\nimport gzip, json, hashlib, math\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nROOT=Path(\'/content/drive/MyDrive/ADNN_Irreversibility/results\')\nPILOT=ROOT/\'scienceworld_physical_pilot_20261107T091929052223Z\'\nRECOVERY=ROOT/\'scienceworld_recovery_diagnostic_20261107T094243161617Z\'\n\ndef index(tree):\n    nodes={}; parents={}\n    def visit(node,parent=None):\n        if not isinstance(node,dict) or \'uuid\' not in node:raise ValueError(\'Unsupported object schema\')\n        uid=str(node[\'uuid\'])\n        if uid in nodes:raise ValueError(\'Duplicate UUID\')\n        nodes[uid]=node;parents[uid]=parent\n        children=node.get(\'contents\',{})\n        if not isinstance(children,dict):raise ValueError(\'Unsupported contents schema\')\n        for c in children.values():visit(c,uid)\n    visit(tree);return nodes,parents\n\ndef descendants(nodes,roots):\n    result=set()\n    def visit(uid):\n        if uid in result or uid not in nodes:return\n        result.add(uid)\n        for c in nodes[uid].get(\'contents\',{}).values():visit(str(c[\'uuid\']))\n    for uid in roots:visit(uid)\n    return result\n\ndef make_scope(before,ids):\n    nodes,parents=index(before)\n    if not ids or any(i not in nodes for i in ids):raise ValueError(\'Binding objects absent\')\n    core=descendants(nodes,ids)\n    context={parents[i] for i in core if parents[i] is not None}-core\n    return {\'roots\':sorted(ids),\'core\':sorted(core),\'context\':sorted(context)}\n\ndef project(tree,scope):\n    nodes,parents=index(tree)\n    ids=set(scope[\'core\'])|set(scope[\'context\'])|descendants(nodes,scope[\'roots\'])\n    result={}\n    for uid in sorted(ids):\n        if uid not in nodes:result[uid]={\'missing\':True};continue\n        n=nodes[uid]\n        result[uid]={\'parent_uuid\':parents[uid],\n          \'properties\':{k:v for k,v in n.items() if k!=\'contents\'},\n          \'children\':sorted(str(c[\'uuid\']) for c in n.get(\'contents\',{}).values())}\n    return result\n\ndef differences(a,b,path=()):\n    if type(a)!=type(b):return [(path,a,b)]\n    if isinstance(a,dict):\n        result=[]\n        for k in sorted(a.keys()|b.keys()):\n            if k not in a or k not in b:result.append((path+(k,),a.get(k),b.get(k)))\n            else:result.extend(differences(a[k],b[k],path+(k,)))\n        return result\n    return [] if a==b else [(path,a,b)]\n\ndef compare(a,b):\n    ds=differences(a,b)\n    def thermal(d):\n        path,x,y=d\n        return path[-2:]==(\'propMaterial\',\'temperatureC\') and all(type(v) in (int,float) and math.isfinite(v) for v in (x,y))\n    return {\'exact_projection_match\':not ds,\'thermal_only_residual\':bool(ds) and all(thermal(d) for d in ds),\n      \'difference_count\':len(ds),\'differences\':[{\'path\':\'/\'.join(p),\'before\':x,\'after\':y} for p,x,y in ds]}\n\ndef main():\n    assert PILOT.is_dir() and RECOVERY.is_dir(),\'Mount Drive; original pilot and recovery directories are required.\'\n    out=ROOT/(\'scienceworld_projection_smoke_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'));out.mkdir()\n    rows=[];errors=[];manifest={}\n    for case in sorted(RECOVERY.glob(\'P[0-9][0-9][0-9]\')):\n        cid=case.name\n        try:\n            binding_path=case/\'action_bindings.json\'\n            bindings=json.loads(binding_path.read_text())[\'bindings\']\n            candidates={tuple(str(i) for i in b[\'obj_ids\']) for b in bindings}\n            if len(candidates)!=1:raise ValueError(\'Missing or ambiguous object-ID binding\')\n            ids=next(iter(candidates))\n            paths=list(PILOT.glob(\'task_*/\'+cid+\'/snapshots.json.gz\'));assert len(paths)==1\n            with gzip.open(paths[0],\'rt\') as f:snap=json.load(f)\n            record=json.loads((case/\'summary.json\').read_text())\n            if record[\'candidate\'].startswith(\'pour \') and len(ids)>=2 and ids[0]==ids[1]:\n                rows.append({\'id\':cid,\'status\':\'excluded_self_pour\',\'final_label\':None});continue\n            scope=make_scope(snap[\'before\'][\'tree\'],set(ids));before=project(snap[\'before\'][\'tree\'],scope)\n            row={\'id\':cid,\'candidate\':record[\'candidate\'],\'scope\':scope,\n                 \'candidate_comparison\':compare(before,project(snap[\'after\'][\'tree\'],scope)),\n                 \'recoveries\':[],\'final_label\':None,\'status\':\'diagnostic_only\'}\n            inputs=[binding_path,paths[0],case/\'summary.json\']\n            for file in sorted(case.glob(\'targeted_*.json.gz\')):\n                with gzip.open(file,\'rt\') as f:trial=json.load(f)\n                row[\'recoveries\'].append({\'undo\':trial[\'test\'][\'undo\'],\n                  \'saved_replay_stable\':trial[\'test\'][\'replay_stable\'],\n                  \'comparison\':compare(before,project(trial[\'end\'][\'tree\'],scope))})\n                inputs.append(file)\n            for file in inputs:manifest[str(file)]=hashlib.sha256(file.read_bytes()).hexdigest()\n            rows.append(row)\n            print(cid,\'recovery comparisons:\',[(t[\'comparison\'][\'exact_projection_match\'],t[\'comparison\'][\'thermal_only_residual\']) for t in row[\'recoveries\']])\n        except Exception as exc:errors.append({\'id\':cid,\'error\':str(exc)})\n    tests=[t for r in rows for t in r.get(\'recoveries\',[])]\n    summary={\'old_cases_reviewed\':len(rows),\'recovery_comparisons\':len(tests),\n       \'exact_projection_matches\':sum(t[\'comparison\'][\'exact_projection_match\'] for t in tests),\n       \'thermal_only_residuals\':sum(t[\'comparison\'][\'thermal_only_residual\'] for t in tests),\n       \'excluded_self_pours\':[r[\'id\'] for r in rows if r[\'status\']==\'excluded_self_pour\'],\n       \'errors\':errors,\'final_labels_created\':0,\'scope_coverage_validated\':False,\n       \'partial_labels_require_exhaustive_search_for_exact_recovery\':True,\n       \'fresh_validation_complete\':False,\'saved_to\':str(out)}\n    for name,data in [(\'summary.json\',summary),(\'results.json\',rows),(\'input_hashes.json\',manifest)]:\n        (out/name).write_text(json.dumps(data,indent=2,allow_nan=False))\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes())\n    print(\'PROJECTED STATE SMOKE TEST COMPLETE\');print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':main()\n')
import runpy
_ = runpy.run_path("/content/39_scienceworld_projection_smoke_colab.py", run_name="__main__")


Send the final PROJECTED STATE SMOKE TEST COMPLETE summary to the project chat. Exact projection matches are not full-world restoration. Thermal-only results are not final partial labels. Next work: review object/property coverage and freeze task-family action manifests before fresh validation.
